# 03 - Xử lý bảng ORDER_ITEMS & ORDER_ITEM_PROMOTION (chuẩn 3NF, tiêu chuẩn Silver)

**Nguồn dữ liệu:** `order_items_silver.csv`
**Bảng đích:**
- `ORDER_ITEMS` — lược đồ 3NF, mục 8
- `ORDER_ITEM_PROMOTION` — lược đồ 3NF, mục 9 (bảng tách ra để xử lý nhóm lặp `promo_id`, vi phạm 1NF ở bản gốc)

**Bảng phụ thuộc:** `ORDER.csv` (đã xuất ở notebook 02) — dùng kiểm tra ràng buộc khóa ngoại `order_id`.

Hai bảng này được xử lý **cùng một notebook** vì có nguồn gốc chung (`order_items_silver.csv`) và quan hệ trực tiếp với nhau (tách 1NF).

| Bảng đích | Thuộc tính | Nguồn | Ghi chú |
|---|---|---|---|
| ORDER_ITEMS | order_id (PK, FK) | order_id | |
| ORDER_ITEMS | product_id (PK, FK) | product_id | |
| ORDER_ITEMS | quantity | quantity | gộp nếu trùng khóa |
| ORDER_ITEMS | unit_price | unit_price | tính lại nếu trùng khóa |
| ORDER_ITEMS | discount_amount | discount_amount | gộp nếu trùng khóa |
| ORDER_ITEM_PROMOTION | order_id (PK, FK) | order_id | |
| ORDER_ITEM_PROMOTION | product_id (PK, FK) | product_id | |
| ORDER_ITEM_PROMOTION | promo_id (PK, FK) | promo_id | chỉ giữ dòng có khuyến mãi thật |

**Cột bị loại bỏ khỏi cả 2 bảng đích:** `total_price_before_discount`, `total_price_after_discount`, `total_price_before_discount_is_outlier`, `total_price_after_discount_is_outlier` — đây là các cột **tính toán/dẫn xuất** (derived), không phải thuộc tính chuẩn hóa theo lược đồ 3NF (có thể tính lại bất kỳ lúc nào từ `quantity * unit_price - discount_amount`). Các cột này chỉ được dùng trong bước kiểm tra chất lượng dữ liệu (QA) bên dưới, không đưa vào bảng xuất cuối.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("./DAAI_N1.4/silver_data_raw")
OUTPUT_DIR = Path("./DAAI_N1.4/silver_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC_FILE = RAW_DIR / "order_items_silver.csv"
ORDER_FILE = OUTPUT_DIR / "ORDER.csv"   # kết quả từ notebook 02
OUT_ITEMS_FILE = OUTPUT_DIR / "ORDER_ITEMS.csv"
OUT_PROMO_FILE = OUTPUT_DIR / "ORDER_ITEM_PROMOTION.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. Nạp dữ liệu & khảo sát chất lượng (Data Profiling)

In [4]:
df_raw = pd.read_csv(SRC_FILE)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (714669, 10)


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,total_price_before_discount,total_price_after_discount,total_price_before_discount_is_outlier,total_price_after_discount_is_outlier
0,1,2400,7,1138.22,0.0,No_Promo,7967.54,7967.54,False,False
1,2,609,7,10166.25,0.0,No_Promo,71163.75,71163.75,True,True
2,3,396,3,11220.33,0.0,No_Promo,33660.99,33660.99,False,False
3,4,635,5,10639.25,0.0,No_Promo,53196.25,53196.25,False,False
4,6,1935,1,1597.84,0.0,No_Promo,1597.84,1597.84,False,False


In [5]:
display(df_raw.dtypes)
print()
print("Null theo cột:")
display(df_raw.isnull().sum())

order_id                                    int64
product_id                                  int64
quantity                                    int64
unit_price                                float64
discount_amount                           float64
promo_id                                   object
total_price_before_discount               float64
total_price_after_discount                float64
total_price_before_discount_is_outlier       bool
total_price_after_discount_is_outlier        bool
dtype: object


Null theo cột:


order_id                                  0
product_id                                0
quantity                                  0
unit_price                                0
discount_amount                           0
promo_id                                  0
total_price_before_discount               0
total_price_after_discount                0
total_price_before_discount_is_outlier    0
total_price_after_discount_is_outlier     0
dtype: int64

In [6]:
# Khóa chính khai báo của ORDER_ITEMS là (order_id, product_id) -> kiểm tra trùng lặp
dup_mask = df_raw.duplicated(subset=['order_id', 'product_id'], keep=False)
n_dup_rows = dup_mask.sum()
n_dup_keys = df_raw[dup_mask][['order_id', 'product_id']].drop_duplicates().shape[0]
print(f"Số dòng liên quan tới khóa (order_id, product_id) bị trùng: {n_dup_rows}")
print(f"Số cặp khóa (order_id, product_id) bị trùng: {n_dup_keys}")
df_raw[dup_mask].sort_values(['order_id', 'product_id']).head(10)

Số dòng liên quan tới khóa (order_id, product_id) bị trùng: 32
Số cặp khóa (order_id, product_id) bị trùng: 16


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,total_price_before_discount,total_price_after_discount,total_price_before_discount_is_outlier,total_price_after_discount_is_outlier
12233,14280,976,1,4019.47,0.00,No_Promo,4019.47,4019.47,False,False
12234,14280,976,2,3937.99,0.00,No_Promo,7875.98,7875.98,False,False
99645,113379,786,6,694.34,0.00,No_Promo,4166.04,4166.04,False,False
99646,113379,786,1,699.37,0.00,No_Promo,699.37,699.37,False,False
189239,215525,1859,5,1896.11,0.00,No_Promo,9480.55,9480.55,False,False
189240,215525,1859,5,1897.20,0.00,No_Promo,9486.00,9486.00,False,False
190291,216740,791,8,793.10,0.00,No_Promo,6344.80,6344.80,False,False
190292,216740,791,5,825.52,0.00,No_Promo,4127.60,4127.60,False,False
214311,243342,777,7,1181.17,1653.64,PROMO-0010,8268.19,6614.55,False,False
214312,243342,777,5,1147.21,1147.21,PROMO-0010,5736.05,4588.84,False,False


**Phát hiện quan trọng:** có 16 cặp `(order_id, product_id)` xuất hiện 2 lần trong dữ liệu gốc, với `quantity`/`unit_price`/`discount_amount` khác nhau giữa 2 dòng (ví dụ: 1 sản phẩm được đặt thành 2 dòng riêng trong cùng 1 đơn — có thể do 2 lần thêm vào giỏ hàng, hoặc lỗi phát sinh ở nguồn). Đây **không phải** là dạng "nhiều khuyến mãi trên 1 dòng hàng" (group lặp promo) mà docx mô tả — vì `promo_id` ở 2 dòng trùng nhau trong hầu hết trường hợp là giống nhau.

**Quyết định xử lý:** vì lược đồ 3NF khai báo khóa chính của `ORDER_ITEMS` là `(order_id, product_id)` — nghĩa là mỗi sản phẩm chỉ được xuất hiện **một dòng duy nhất** trong một đơn hàng — nên các cặp trùng này sẽ được **gộp (aggregate)**:
- `quantity` → cộng dồn (tổng số lượng thực tế đã đặt)
- `discount_amount` → cộng dồn (tổng tiền giảm giá thực tế)
- `unit_price` → tính lại theo **bình quân gia quyền** theo số lượng: `sum(quantity Ã— unit_price) / sum(quantity)`, để tổng giá trị đơn hàng trước khi gộp và sau khi gộp là tương đương nhau.

Việc trích xuất `ORDER_ITEM_PROMOTION` sẽ được thực hiện **trước khi gộp**, dựa trên dữ liệu gốc (chưa gộp) để không bỏ sót trường hợp 2 dòng trùng khóa có `promo_id` khác nhau (nếu có).

In [7]:
# Kiểm tra công thức tính toán nội bộ của nguồn (để tin cậy vào discount_amount/quantity/unit_price)
calc_before = (df_raw['quantity'] * df_raw['unit_price']).round(2)
diff_before = (calc_before - df_raw['total_price_before_discount']).abs().max()
calc_after = (df_raw['total_price_before_discount'] - df_raw['discount_amount']).round(2)
diff_after = (calc_after - df_raw['total_price_after_discount']).abs().max()
print(f"Sai lệch tối đa total_price_before_discount vs quantity*unit_price: {diff_before}")
print(f"Sai lệch tối đa total_price_after_discount vs (before - discount): {diff_after}")
print("=> Sai lệch chỉ do làm tròn số thập phân, công thức nội bộ nhất quán.")

Sai lệch tối đa total_price_before_discount vs quantity*unit_price: 0.010000000009313226
Sai lệch tối đa total_price_after_discount vs (before - discount): 0.010000000009313226
=> Sai lệch chỉ do làm tròn số thập phân, công thức nội bộ nhất quán.


In [8]:
# Khảo sát các cột còn lại
print("quantity:", df_raw['quantity'].min(), "-", df_raw['quantity'].max())
print("unit_price:", df_raw['unit_price'].min(), "-", df_raw['unit_price'].max())
print("discount_amount:", df_raw['discount_amount'].min(), "-", df_raw['discount_amount'].max())
print("Số giá trị âm ở quantity/unit_price/discount_amount:",
      (df_raw['quantity'] <= 0).sum(), (df_raw['unit_price'] < 0).sum(), (df_raw['discount_amount'] < 0).sum())
print()
print("promo_id: số giá trị duy nhất =", df_raw['promo_id'].nunique())
print("Tỷ lệ dòng không có khuyến mãi (No_Promo):",
      f"{(df_raw['promo_id'] == 'No_Promo').mean():.1%}")
print()
print("Số dòng được đánh dấu outlier (total_before):", df_raw['total_price_before_discount_is_outlier'].sum())
print("Số dòng được đánh dấu outlier (total_after):", df_raw['total_price_after_discount_is_outlier'].sum())

quantity: 1 - 8
unit_price: 392.57 - 15324.065
discount_amount: 0.0 - 2419.075
Số giá trị âm ở quantity/unit_price/discount_amount: 0 0 0

promo_id: số giá trị duy nhất = 51
Tỷ lệ dòng không có khuyến mãi (No_Promo): 61.3%

Số dòng được đánh dấu outlier (total_before): 36399
Số dòng được đánh dấu outlier (total_after): 36494


**Nhận xét khảo sát:**
- Không có null, không có `quantity`/`unit_price`/`discount_amount` âm hoặc bằng 0 bất thường.
- Công thức `total_price_before/after_discount` nhất quán với `quantity`, `unit_price`, `discount_amount` (sai lệch chỉ do làm tròn) → có thể **loại bỏ an toàn** 2 cột `total_price_*` vì tính lại được, đúng tinh thần chuẩn hóa 3NF (loại bỏ thuộc tính suy diễn được).
- Cờ `*_is_outlier` là kết quả một bước phát hiện ngoại lai đã có sẵn ở tầng trước — không thuộc lược đồ chuẩn hóa nên không đưa vào bảng đích, nhưng được ghi nhận số lượng ở đây để đối chiếu nếu cần điều tra thêm ở bước phân tích sau này.
- 16 cặp khóa trùng lặp là vấn đề chất lượng dữ liệu cần xử lý (chi tiết ở trên).

## 2. Tách bảng ORDER_ITEM_PROMOTION (xử lý vi phạm 1NF)

In [9]:
# Chỉ giữ các dòng thực sự có áp dụng khuyến mãi (loại bỏ giá trị placeholder 'No_Promo')
promo_df = (
    df_raw.loc[df_raw['promo_id'] != 'No_Promo', ['order_id', 'product_id', 'promo_id']]
    .drop_duplicates()
    .copy()
)

promo_df['order_id'] = promo_df['order_id'].astype('int64')
promo_df['product_id'] = promo_df['product_id'].astype('int64')
promo_df['promo_id'] = promo_df['promo_id'].str.strip()

promo_df = promo_df.sort_values(['order_id', 'product_id', 'promo_id']).reset_index(drop=True)

print("Số dòng ORDER_ITEM_PROMOTION:", len(promo_df))
promo_df.head()

Số dòng ORDER_ITEM_PROMOTION: 276309


,order_id,product_id,promo_id
0,46253,2250,PROMO-0006
1,46254,2251,PROMO-0006
2,46257,785,PROMO-0006
3,46257,786,PROMO-0006
4,46258,1093,PROMO-0006


In [10]:
# Kiểm tra khóa chính (order_id, product_id, promo_id) của ORDER_ITEM_PROMOTION
n_dup_promo_pk = promo_df.duplicated(subset=['order_id', 'product_id', 'promo_id']).sum()
print(f"Số dòng trùng khóa chính trong ORDER_ITEM_PROMOTION: {n_dup_promo_pk}")
assert n_dup_promo_pk == 0, "Vi phạm khóa chính (order_id, product_id, promo_id)!"

# Với dữ liệu hiện tại, mỗi dòng hàng chỉ có tối đa 1 khuyến mãi (raw chỉ có 1 cột promo_id, không có promo_id_2)
max_promo_per_item = promo_df.groupby(['order_id', 'product_id']).size().max()
print(f"Số khuyến mãi tối đa trên 1 dòng hàng (order_id, product_id) trong dữ liệu hiện tại: {max_promo_per_item}")
print("=> Cấu trúc bảng vẫn hỗ trợ N khuyến mãi/dòng hàng nếu phát sinh trong tương lai, dù dữ liệu hiện tại chỉ có tối đa 1.")

Số dòng trùng khóa chính trong ORDER_ITEM_PROMOTION: 0
Số khuyến mãi tối đa trên 1 dòng hàng (order_id, product_id) trong dữ liệu hiện tại: 1
=> Cấu trúc bảng vẫn hỗ trợ N khuyến mãi/dòng hàng nếu phát sinh trong tương lai, dù dữ liệu hiện tại chỉ có tối đa 1.


## 3. Gộp dòng trùng khóa & chuyển đổi bảng ORDER_ITEMS theo lược đồ 3NF

In [11]:
df = df_raw.copy()
df['order_id'] = df['order_id'].astype('int64')
df['product_id'] = df['product_id'].astype('int64')

# Gộp theo khóa chính (order_id, product_id):
# - quantity: cộng dồn
# - discount_amount: cộng dồn
# - unit_price: bình quân gia quyền theo quantity (bảo toàn tổng giá trị trước giảm giá)
def aggregate_group(g):
    total_qty = g['quantity'].sum()
    weighted_unit_price = round((g['quantity'] * g['unit_price']).sum() / total_qty, 2)
    return pd.Series({
        'quantity': int(total_qty),
        'unit_price': weighted_unit_price,
        'discount_amount': round(g['discount_amount'].sum(), 2),
    })

items_df = (
    df.groupby(['order_id', 'product_id'], as_index=False)
      .apply(aggregate_group, include_groups=False)
)

print("Số dòng trước khi gộp:", len(df))
print("Số dòng sau khi gộp:", len(items_df))
print("Số dòng đã được gộp lại (giảm đi):", len(df) - len(items_df))

Số dòng trước khi gộp: 714669
Số dòng sau khi gộp: 714653
Số dòng đã được gộp lại (giảm đi): 16


In [12]:
# Sắp xếp cột đúng theo lược đồ ORDER_ITEMS
items_df = items_df[['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount']]
# groupby.apply có thể ép quantity thành float64 khi gộp -> ép lại đúng kiểu integer(10) theo lược đồ
items_df['quantity'] = items_df['quantity'].astype('int64')
items_df = items_df.sort_values(['order_id', 'product_id']).reset_index(drop=True)
items_df.head()

,order_id,product_id,quantity,unit_price,discount_amount
0,1,2400,7,1138.22,0.0
1,2,609,7,10166.25,0.0
2,3,396,3,11220.33,0.0
3,4,635,5,10639.25,0.0
4,6,1935,1,1597.84,0.0


## 4. Kiểm tra ràng buộc cuối & xuất 2 bảng Silver chuẩn hóa

In [13]:
# (a) Khóa chính ORDER_ITEMS phải duy nhất tuyệt đối sau khi gộp
n_dup_pk_final = items_df.duplicated(subset=['order_id', 'product_id']).sum()
print(f"Số cặp (order_id, product_id) còn trùng sau khi gộp: {n_dup_pk_final}")
assert n_dup_pk_final == 0, "Vẫn còn vi phạm khóa chính (order_id, product_id) sau khi gộp!"

# (b) Không còn null ở bất kỳ cột nào
for col in items_df.columns:
    assert items_df[col].isnull().sum() == 0, f"Cột {col} còn giá trị null!"

# (c) Kiểm tra ràng buộc khóa ngoại order_id với bảng ORDER đã xử lý ở notebook 02
order_ids = pd.read_csv(ORDER_FILE, usecols=['order_id'])['order_id']
orphan_items = (~items_df['order_id'].isin(order_ids)).sum()
orphan_promo = (~promo_df['order_id'].isin(order_ids)).sum()
print(f"ORDER_ITEMS có order_id không tồn tại trong ORDER: {orphan_items}")
print(f"ORDER_ITEM_PROMOTION có order_id không tồn tại trong ORDER: {orphan_promo}")
assert orphan_items == 0 and orphan_promo == 0, "Phát hiện order_id mồ côi — cần xử lý trước khi nạp!"

Số cặp (order_id, product_id) còn trùng sau khi gộp: 0
ORDER_ITEMS có order_id không tồn tại trong ORDER: 0
ORDER_ITEM_PROMOTION có order_id không tồn tại trong ORDER: 0


In [14]:
# (d) Đối chiếu tổng giá trị trước/sau khi gộp để đảm bảo không thất thoát dữ liệu
total_qty_before = df_raw['quantity'].sum()
total_qty_after = items_df['quantity'].sum()
total_value_before = (df_raw['quantity'] * df_raw['unit_price']).sum()
total_value_after = (items_df['quantity'] * items_df['unit_price']).sum()

print(f"Tổng quantity trước gộp: {total_qty_before:,} | sau gộp: {total_qty_after:,}")
print(f"Tổng giá trị (quantity*unit_price) trước gộp: {total_value_before:,.2f}")
print(f"Tổng giá trị (quantity*unit_price) sau gộp:  {total_value_after:,.2f}")
print(f"Chênh lệch: {abs(total_value_before - total_value_after):,.2f}")

assert total_qty_before == total_qty_after, "Tổng quantity bị thay đổi sau khi gộp!"

Tổng quantity trước gộp: 3,213,143 | sau gộp: 3,213,143
Tổng giá trị (quantity*unit_price) trước gộp: 16,323,582,199.08
Tổng giá trị (quantity*unit_price) sau gộp:  16,323,582,004.57
Chênh lệch: 194.51


In [15]:
print("=== ORDER_ITEMS ===")
print("Số dòng:", len(items_df), "| Số cột:", len(items_df.columns))
display(items_df.dtypes)
print()
print("=== ORDER_ITEM_PROMOTION ===")
print("Số dòng:", len(promo_df), "| Số cột:", len(promo_df.columns))
display(promo_df.dtypes)

=== ORDER_ITEMS ===
Số dòng: 714653 | Số cột: 5


order_id             int64
product_id           int64
quantity             int64
unit_price         float64
discount_amount    float64
dtype: object


=== ORDER_ITEM_PROMOTION ===
Số dòng: 276309 | Số cột: 3


order_id       int64
product_id     int64
promo_id      object
dtype: object

In [16]:
items_df.to_csv(OUT_ITEMS_FILE, index=False)
promo_df.to_csv(OUT_PROMO_FILE, index=False)

print(f"Đã xuất bảng ORDER_ITEMS tại: {OUT_ITEMS_FILE.resolve()}")
print(f"Đã xuất bảng ORDER_ITEM_PROMOTION tại: {OUT_PROMO_FILE.resolve()}")

Đã xuất bảng ORDER_ITEMS tại: D:\TH_DA&AI\DAAI_N1.4\silver_data\ORDER_ITEMS.csv
Đã xuất bảng ORDER_ITEM_PROMOTION tại: D:\TH_DA&AI\DAAI_N1.4\silver_data\ORDER_ITEM_PROMOTION.csv
